In [ ]:
import os

import numpy as np
import pandas as pd
import umap
from pykalman import KalmanFilter
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import cv2

In [2]:
video_files = [video.name for video in os.scandir('data') if video.is_dir()]

X_train_temp, X_test = train_test_split(video_files, test_size=0.15, random_state=42)
X_train, X_val = train_test_split(X_train_temp, test_size=0.1765, random_state=42)

In [3]:
def get_bounding_box(filename):
    with open(filename) as f:
        data = f.readlines()

    combined = ' '.join(data)
    start = combined.index("{") + 3
    end = combined.index("}")
    points = [coord.split(" ") for coord in combined[start:end].strip().split("\n ")]
    points = np.array(points).astype(float)

    max_coords = np.max(points, axis=0)
    min_coords = np.min(points, axis=0)
    x, y = min_coords
    w, h = max_coords - min_coords
    
    return x, y, w, h

def get_true_states(video_id):
    filename = f"data/{video_id}/annot"
    frames = sorted([file for file in os.listdir(filename) if file.endswith(".pts")])
    Z = []

    for frame in frames:
        file = filename + "/" + frame
        x, y, w, h = get_bounding_box(file)
        Z.append([x, y, w, h])

    return np.array(Z).T

def get_observations(video_id, force_recompute=False):
    cache_path = f"data/{video_id}/observations.npy"
    none_mask_path = f"data/{video_id}/none_mask.npy"

    # Return cached result if it exists
    if os.path.exists(cache_path) and not force_recompute:
        Z = np.load(cache_path)
        none_mask = np.load(none_mask_path)
        return Z, none_mask

    # Load detector
    detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    cap = cv2.VideoCapture(f"data/{video_id}/vid.avi")

    observations = []
    none_mask = []

    while True:
        ret, frame = cap.read()
        if not ret or frame is None or frame.size == 0:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        detections = detector.detectMultiScale(gray,
                                               scaleFactor=1.1,
                                               minNeighbors=5,
                                               minSize=(60, 60))

        if len(detections) > 0:
            x, y, w, h = detections[0]
            observations.append([x, y, w, h])
            none_mask.append(False)
        else:
            observations.append([0, 0, 0, 0])
            none_mask.append(True)

    cap.release()

def get_continuous_chunks(video_id):
    Z, none_mask = get_observations(video_id)

    split_indices = np.where(none_mask)[0]
    segments = np.split(Z.T, split_indices)
    cleaned_segments = [segments[0]] + [seg[1:] for seg in segments[1:]]

    return cleaned_segments

def compute_log_likelihood(segments, kf):
    """Compute average log-likelihood per frame across all validation segments."""
    total_ll = 0
    total_frames = 0
    for seq in segments:
        try:
            ll = kf.loglikelihood(seq)  # one value per frame
            total_ll += ll.sum()
            total_frames += seq.shape[0]
        except ValueError:
            continue
    return total_ll / total_frames if total_frames > 0 else -np.inf

In [4]:
MIN_LENGTH = 10

train_segments = []
for video_id in X_train:
    chunks = get_continuous_chunks(video_id)
    train_segments.extend([c for c in chunks if c.shape[0] >= MIN_LENGTH])

val_segments = []
for video_id in X_val:
    chunks = get_continuous_chunks(video_id)
    val_segments.extend([c for c in chunks if c.shape[0] >= MIN_LENGTH])

# Reduce 4-dimensional bounding box data to 2 dimensions for training via UMAP

In [5]:
all_obs_flat = np.vstack(train_segments)

reducer = umap.UMAP(n_components=2, random_state=42)
reducer.fit(all_obs_flat)

/home/connor-mcbride/acme/math404/final-project/em-kalman-face-tracker/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/connor-mcbride/acme/math404/final-project/em-kalman-face-tracker/.venv/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


,n_neighbors,15
,n_components,2
,metric,'euclidean'
,metric_kwds,None
,output_metric,'euclidean'
,output_metric_kwds,None
,n_epochs,None
,learning_rate,1.0
,init,'spectral'
,min_dist,0.1
,spread,1.0


In [6]:
train_latent = []
val_latent = []

for s in tqdm(train_segments):
    train_latent.append(reducer.transform(s))

for s in tqdm(val_segments):
    val_latent.append(reducer.transform(s))

100%|██████████| 141/141 [00:08<00:00, 16.45it/s]


In [7]:
F = np.array([
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
])

H = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0]
])

n_dim_state = 4
n_dim_obs = 2

In [ ]:
best_ll = -np.inf 
best_Q = None
best_R = None
patience = 5  # stop if no improvement for this many iterations
no_improve = 0

Q_est = 0.01 * np.eye(4)
R_est = 0.05 * np.eye(2)  # Smaller initial guesses since we are no longer is pixel space

n_em_iters = 20

for em_iter in tqdm(range(n_em_iters)):
    Q_accum = np.zeros((n_dim_state, n_dim_state))
    R_accum = np.zeros((n_dim_obs, n_dim_obs))
    total_frames = 0

    for seq in tqdm(train_latent):
        T = len(seq)
        init_mean = np.concatenate([seq[0], [0, 0]])
        init_cov = np.eye(n_dim_state) * 0.1

        kf_seq = KalmanFilter(
            transition_matrices=F,
            observation_matrices=H,
            transition_covariance=Q_est,
            observation_covariance=R_est,
            initial_state_mean=init_mean,
            initial_state_covariance=init_cov,
            n_dim_state=n_dim_state,
            n_dim_obs=n_dim_obs
        )

        try:
            kf_updated = kf_seq.em(seq, n_iter=5,
                                   em_vars=['transition_covariance', 
                                            'observation_covariance'])
            
            # Weight contribution by segment length
            Q_accum += T * kf_updated.transition_covariance
            R_accum += T * kf_updated.observation_covariance
            total_frames += T
        except:
            continue  # skip segments that cause numerical issues

    Q_est = Q_accum / total_frames
    R_est = R_accum / total_frames

    # Evaluate on validation set
    val_ll_total = 0
    total_val_frames = 0

    for seq in val_latent:
        init_mean_val = np.concatenate([seq[0], [0, 0]])

        kf_val = KalmanFilter(
            transition_matrices=F,
            observation_matrices=H,
            transition_covariance=Q_est,
            observation_covariance=R_est,
            initial_state_mean=init_mean_val,
            initial_state_covariance=np.eye(n_dim_state) * 0.1,
            n_dim_state=n_dim_state,
            n_dim_obs=n_dim_obs
        )

        val_ll_total += kf_val.loglikelihood(seq)
        total_val_frames += len(seq)

    current_val_ll = val_ll_total / total_val_frames
    print(f"EM iter {em_iter+1} — val log-likelihood: {current_val_ll:.4f}")

    if current_val_ll > best_ll:
        best_ll = current_val_ll
        best_Q = Q_est.copy()
        best_R = R_est.copy()
        no_improve = 0
        print("New best — saving Q and R")
    else:
        no_improve += 1
        print(f"  No improvement ({no_improve}/{patience})")
        if no_improve >= patience:
            print("Early stopping triggered")
            break

# Use best parameters found
Q_est = best_Q
R_est = best_R
print("Estimated Q diag:", np.diag(Q_est).round(4))
print("Estimated R diag:", np.diag(R_est).round(4))

  5%|▌         | 1/20 [04:49<1:31:34, 289.19s/it]

EM iter 1 — val log-likelihood: -5.8427
New best — saving Q and R


In [28]:
np.save('parameters/R_latent.npy', R_est)
np.save('parameters/Q_latent.npy', Q_est)
print("Saved R and Q")

Saved R and Q


# Evaluation on Test Set

In [19]:
def compute_rmse(x_hat, z_gt):
    """
    x_hat : (T, 4) estimated states [x, y, w, h]
    z_gt  : (T, 4) ground truth bounding boxes
    """
    return np.sqrt(np.mean((x_hat - z_gt) ** 2, axis=0))

def compute_iou(box1, box2):
    """
    box1, box2 : arrays of [x, y, w, h]
    """
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2 = min(box1[0] + box1[2], box2[0] + box2[2])
    y2 = min(box1[1] + box1[3], box2[1] + box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = box1[2] * box1[3]
    area2 = box2[2] * box2[3]
    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0.0

def compute_mean_iou(x_hat, z_gt):
    """
    x_hat : (T, 4)
    z_gt  : (T, 4)
    """
    ious = [compute_iou(x_hat[t], z_gt[t]) for t in range(len(x_hat))]
    return np.mean(ious)

def compute_nees(x_hat, P_hat, z_gt, H):
    """
    x_hat : (T, 8) estimated states
    P_hat : (T, 8, 8) estimated covariances
    z_gt  : (T, 4) ground truth
    H     : (4, 8) observation matrix
    """
    T = x_hat.shape[0]
    nees_values = []

    for t in range(T):
        error = z_gt[t] - H @ x_hat[t]          # innovation against ground truth
        S = H @ P_hat[t] @ H.T                   # projected covariance
        nees_t = error @ np.linalg.solve(S, error)
        nees_values.append(nees_t)

    return np.mean(nees_values)  # should be ~4 if well calibrated

def compute_nis(x_hat_prior, P_hat_prior, z_obs, H, R):
    """
    x_hat_prior : (T, 8) predicted states before update
    P_hat_prior : (T, 8, 8) predicted covariances before update
    z_obs       : (T, 4) actual detector observations
    """
    T = z_obs.shape[0]
    nis_values = []

    for t in range(T):
        innovation = z_obs[t] - H @ x_hat_prior[t]
        S = H @ P_hat_prior[t] @ H.T + R
        nis_t = innovation @ np.linalg.solve(S, innovation)
        nis_values.append(nis_t)

    return np.mean(nis_values)

In [23]:
results = []
Q_est = np.load('parameters/Q_latent.npy')
R_est = np.load('parameters/R_latent.npy')

for video_id in tqdm(X_test):
    Z_raw, none_mask = get_observations(video_id)
    Z_gt = get_true_states(video_id).T  # (T, 4)

    # Only use frames where detector found a face
    detected = ~none_mask
    Z_obs = Z_raw.T          # (T, 4)
    Z_valid = Z_obs[detected]    # only detected frames
    gt_valid = Z_gt[detected]    # matching ground truth frames

    if len(Z_valid) < 2:
        print(f"Video {video_id} — too few detections, skipping")
        continue

    Z_latent = reducer.transform(Z_valid)

    init_mean_val = np.concatenate([Z_latent[0], [0, 0]])

    kf = KalmanFilter(
        transition_matrices=F,
        observation_matrices=H,
        transition_covariance=Q_est,
        observation_covariance=R_est,
        initial_state_mean=init_mean_val,
        initial_state_covariance=np.eye(n_dim_state) * 0.1,
        n_dim_state=n_dim_state,
        n_dim_obs=n_dim_obs
    )

    try:
        smoothed_means, smoothed_covs = kf.filter(Z_latent)
    except ValueError:
        print(f"Video {video_id} — filter failed, skipping")
        continue

    latent_positions = smoothed_means[:, :2]
    x_hat = reducer.inverse_transform(latent_positions)

    if not np.all(np.isfinite(x_hat)):
        print(f"Video {video_id} — non-finite values in filter output, skipping")
        continue

    gt_latent = reducer.transform(gt_valid)

    nees = compute_nees(smoothed_means, smoothed_covs, gt_latent, H)
    rmse = compute_rmse(x_hat, gt_valid).mean()
    miou = compute_mean_iou(x_hat, gt_valid)

    results.append({
        'video_id':  video_id,
        'rmse':      rmse,
        'mean_iou':  miou,
        'nees':      nees,
        'n_frames':  len(Z_valid)
    })

    print(f"Video {video_id} — mIoU: {miou:.3f}, RMSE: {rmse:.2f}, "
          f"NEES: {nees:.2f}, frames: {len(Z_valid)}")

df = pd.DataFrame(results)
summary = {
    'video_id': 'MEAN',
    'rmse':     df['rmse'].mean(),
    'mean_iou': df['mean_iou'].mean(),
    'nees':     df['nees'].mean(),
    'n_frames': df['n_frames'].sum()
}

print("\n--- Test Set Summary ---")
print(f"Mean RMSE:  {summary['rmse']:.3f}")
print(f"Mean IoU:   {summary['mean_iou']:.3f}")
print(f"Mean NEES:  {summary['nees']:.3f}")
print(f"Total frames evaluated: {summary['n_frames']}")

df = pd.concat([df, pd.DataFrame([summary])], ignore_index=True)
df.to_csv('results/latent_results.csv', index=False)

  6%|▌         | 1/18 [00:44<12:41, 44.81s/it]

Video 562 — mIoU: 0.327, RMSE: 107.29, NEES: 35.47, frames: 1310


 11%|█         | 2/18 [01:26<11:30, 43.15s/it]

Video 521 — mIoU: 0.152, RMSE: 117.98, NEES: 24.99, frames: 863


 17%|█▋        | 3/18 [02:11<10:57, 43.86s/it]

...

Video 160 — mIoU: 0.234, RMSE: 118.05, NEES: 16.56, frames: 2997


 94%|█████████▍| 17/18 [12:55<00:47, 47.34s/it]

Video 019 — mIoU: 0.320, RMSE: 93.06, NEES: 10.33, frames: 1769


100%|██████████| 18/18 [13:43<00:00, 45.77s/it]

Video 506 — mIoU: 0.243, RMSE: 96.05, NEES: 5.89, frames: 1775

--- Test Set Summary ---
Mean RMSE:  105.112
Mean IoU:   0.269
Mean NEES:  22.897
Total frames evaluated: 26868
